# ECE daily-mean audit (ece-daily-mean-1.0)

# ECE daily-mean audit (ece-daily-mean-1.0)

This notebook audits whether the 5 ECE raw files cover a full 24-hour day, compares daily-mean definitions against the USCRN daily01 equal-hour weighting, and documents that SNOTEL `.stm` files arrive hourly (cross-sensor averaged in `snotel_pipe.py`, not daily-averaged). All reusable logic lives in `audit_coverage.py`, `compare_averaging.py`, and `snotel_check.py` in this directory for reproducibility.

In [1]:
from pathlib import Path
import sys
import yaml
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR.parents[2]
EXP_DIR = NB_DIR
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))
print(f"NB_DIR={NB_DIR}")
print(f"REPO_ROOT={REPO_ROOT}")
with open(EXP_DIR / "config.yaml") as f:
    CFG = yaml.safe_load(f)
print("stations:", list(CFG["ece_files"].keys()))
FIG_DIR = EXP_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)


NB_DIR=/scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/ece-daily-mean-1.0
REPO_ROOT=/scratch/group/p.cis250607.000/MDR-Project
stations: ['ECE_BBG_Main_St', 'ECE_BBG_Lost_Meadow', 'ECE_Renton_Home', 'ECE_Renton_Garden_North', 'ECE_Renton_Garden_Shed']


In [2]:
from audit_coverage import load_ece_raw, daily_coverage, hourly_matrix
from compare_averaging import daily_estimators, summarize_deltas
from snotel_check import load_stm_hourly, daily_reductions, find_sample_stm
print("audit modules imported")

audit modules imported


## ECE 24-hour coverage

We load all 5 raw files with Seattle Time, floor to calendar day, and count samples per day and per hour. A true 24-hour daily mean needs samples in all 24 hours; we flag days with fewer than 18 distinct hours and check the known `2026-08-01` gap plus partial edge days.

In [3]:
ece_dir = REPO_ROOT / CFG["repo_paths"]["ece_raw_dir"]
coverage_rows = []
hour_mats = {}
raw_store = {}
for station, fname in CFG["ece_files"].items():
    df = load_ece_raw(ece_dir / fname)
    raw_store[station] = df
    cov = daily_coverage(df)
    cov["station"] = station
    coverage_rows.append(cov)
    hour_mats[station] = hourly_matrix(df)
    print(f"{station}: {len(df)} samples, {df['date'].min().date()}..{df['date'].max().date()}, days={len(cov)}")
coverage = pd.concat(coverage_rows, ignore_index=True)
eval_start, eval_end = pd.to_datetime("2026-07-20"), pd.to_datetime("2026-08-19")
eval_cov = coverage[(coverage["date"] >= eval_start) & (coverage["date"] <= eval_end)].copy()
print(f"EVAL days present: {len(eval_cov)} (expected 150 = 30d x 5 stations)")
print(f"missing eval dates per station:")
for station, sub in eval_cov.groupby("station"):
    got = set(sub["date"].dt.strftime("%Y-%m-%d"))
    want = set(pd.date_range(eval_start, eval_end, freq="D").strftime("%Y-%m-%d"))
    print(f"  {station}: missing={sorted(want - got)}")

ECE_BBG_Main_St: 13600 samples, 2026-07-19..2026-08-20, days=32
ECE_BBG_Lost_Meadow: 12427 samples, 2026-07-19..2026-08-20, days=32


ECE_Renton_Home: 13126 samples, 2026-07-19..2026-08-20, days=32
ECE_Renton_Garden_North: 12311 samples, 2026-07-19..2026-08-20, days=32


ECE_Renton_Garden_Shed: 9132 samples, 2026-07-19..2026-08-20, days=32
EVAL days present: 150 (expected 150 = 30d x 5 stations)
missing eval dates per station:


  ECE_BBG_Lost_Meadow: missing=['2026-08-01']
  ECE_BBG_Main_St: missing=['2026-08-01']
  ECE_Renton_Garden_North: missing=['2026-08-01']
  ECE_Renton_Garden_Shed: missing=['2026-08-01']
  ECE_Renton_Home: missing=['2026-08-01']


In [4]:
min_hours = int(CFG["params"]["min_hours_per_day"])
flag = eval_cov[eval_cov["n_hours"] < min_hours].copy()
print(f"days with n_hours < {min_hours}: {len(flag)}")
print(eval_cov.groupby("station")[["n_samples", "n_hours"]].agg(["min", "median", "max"]).to_string())
print("worst 5 low-hour days:")
print(eval_cov.sort_values("n_hours").head(5).to_string(index=False))
eval_cov.to_csv(EXP_DIR / "ece_eval_coverage.csv", index=False)
print("saved ece_eval_coverage.csv")

days with n_hours < 18: 3
                        n_samples             n_hours           
                              min median  max     min median max
station                                                         
ECE_BBG_Lost_Meadow           241  378.5  624      21   24.0  24
ECE_BBG_Main_St               392  428.0  518      24   24.0  24
ECE_Renton_Garden_North       278  408.5  510      16   24.0  24
ECE_Renton_Garden_Shed        183  302.0  383      16   24.0  24
ECE_Renton_Home               266  431.5  484      16   24.0  24
worst 5 low-hour days:
      date  n_samples  n_hours          first_time           last_time                 station
2026-07-28        283       16 2026-07-28 00:06:03 2026-07-28 23:48:46 ECE_Renton_Garden_North
2026-07-28        266       16 2026-07-28 00:00:14 2026-07-28 23:45:52         ECE_Renton_Home
2026-07-28        183       16 2026-07-28 00:27:16 2026-07-28 23:50:20  ECE_Renton_Garden_Shed
2026-08-19        278       18 2026-08-19 00:04:42 

saved ece_eval_coverage.csv


## Daily-mean definitions

We compare the current `simple mean of all sub-minute samples` against an `hourly-weighted mean` (mean of hourly means, mirroring USCRN equal-hour weighting), plus `median` and `midnight snapshot` as stress tests. Deltas are in fraction m3/m3.

In [5]:
est_rows = []
for station, df in raw_store.items():
    est = daily_estimators(df)
    est["station"] = station
    est_rows.append(est)
ests = pd.concat(est_rows, ignore_index=True)
ests_eval = ests[(ests["date"] >= eval_start) & (ests["date"] <= eval_end)].copy()
delta_summary = (
    ests_eval.groupby("station")
    .apply(lambda d: summarize_deltas(d).assign(station=d.name), include_groups=False)
    .reset_index(drop=True)
)
print(delta_summary.to_string(index=False))
ests_eval.to_csv(EXP_DIR / "ece_daily_estimators.csv", index=False)
print("saved ece_daily_estimators.csv")

                 comparison  mean_abs_delta  max_abs_delta                 station
  simple vs hourly_weighted        0.001269       0.003627     ECE_BBG_Lost_Meadow
           simple vs median        0.004010       0.021777     ECE_BBG_Lost_Meadow
simple vs midnight_snapshot        0.009604       0.021377     ECE_BBG_Lost_Meadow
  simple vs hourly_weighted        0.000301       0.000820         ECE_BBG_Main_St
           simple vs median        0.002138       0.003507         ECE_BBG_Main_St
simple vs midnight_snapshot        0.007311       0.012391         ECE_BBG_Main_St
  simple vs hourly_weighted        0.000643       0.002152 ECE_Renton_Garden_North
           simple vs median        0.004197       0.015702 ECE_Renton_Garden_North
simple vs midnight_snapshot        0.019069       0.038919 ECE_Renton_Garden_North
  simple vs hourly_weighted        0.000487       0.001318  ECE_Renton_Garden_Shed
           simple vs median        0.002223       0.006460  ECE_Renton_Garden_Shed
simp

In [6]:
fig, ax = plt.subplots(figsize=(8, 4))
for station, sub in ests_eval.groupby("station"):
    sub = sub.sort_values("date")
    ax.plot(sub["date"], sub["simple_minus_hourly"] * 100, marker="o", ms=3, label=station)
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("simple - hourly-weighted (pp)")
ax.set_title("ECE: simple mean vs hourly-weighted mean")
ax.legend(fontsize=7, ncol=2)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "simple_vs_hourly.png", dpi=150)
print("saved simple_vs_hourly.png")
fig2, ax2 = plt.subplots(figsize=(8, 3.5))
full_days = pd.date_range(eval_start, eval_end, freq="D")
mat = hour_mats["ECE_Renton_Home"].reindex(full_days, fill_value=0)
im = ax2.imshow(mat.values, aspect="auto", interpolation="nearest")
ax2.set_xlabel("hour of day (Seattle Time)")
ax2.set_ylabel("eval day (0=Jul-20, 30=Aug-19; Aug-01 row = 0)")
ax2.set_title("Renton Home: samples per hour (eval window)")
fig2.colorbar(im, label="n samples")
fig2.tight_layout()
fig2.savefig(FIG_DIR / "renton_home_hourly.png", dpi=150)
print("saved renton_home_hourly.png")


saved simple_vs_hourly.png


saved renton_home_hourly.png


## SNOTEL hourly vs daily

SNOTEL `.stm` files store hourly rows. `snotel_pipe.py` averages redundant sensors at the same timestamp but keeps hourly granularity. We verify this on one 5 cm SourdoughGulch file and compare `daily mean of 24 hourly values` against a `first-hour snapshot`, which is what `MergePipe.drop_duplicates(["station_id","date"])` would retain if run on hourly data.

In [7]:
snotel_dir = REPO_ROOT / CFG["repo_paths"]["snotel_raw_dirs"][0]
stm_path = find_sample_stm(snotel_dir, CFG["params"]["snotel_sample_station"] and CFG["params"]["snotel_sample_depth"])
print(f"sample stm: {stm_path.name if stm_path else None}")
hourly = load_stm_hourly(stm_path)
print(f"hourly rows: {len(hourly)}, span {hourly['DateTime'].min()}..{hourly['DateTime'].max()}")
print("first 3 rows:", hourly[["DateTime", "Value"]].head(3).to_dict("records"))
daily = daily_reductions(hourly)
jan = daily[(daily["date"] >= "2017-01-01") & (daily["date"] <= "2017-01-07")]
print(jan.to_string(index=False))
print(f"Jan2017 mean|mean-first|: mean={daily['mean_minus_first'].abs().mean():.4f}, max={daily['mean_minus_first'].abs().max():.4f}")
daily.to_csv(EXP_DIR / "snotel_sample_daily.csv", index=False)
print("saved snotel_sample_daily.csv")

sample stm: SNOTEL_SNOTEL_SourdoughGulch_sm_0.050800_0.050800_Stevens-Hydraprobe-Analog_2_1_20160101_20260824.stm


hourly rows: 58199, span 2016-01-01 00:00:00..2022-08-23 07:00:00
first 3 rows: [{'DateTime': Timestamp('2016-01-01 00:00:00'), 'Value': 0.292}, {'DateTime': Timestamp('2016-01-01 01:00:00'), 'Value': 0.292}, {'DateTime': Timestamp('2016-01-01 02:00:00'), 'Value': 0.292}]
      date  n_hours  daily_mean  first_hour  mean_minus_first
2017-01-01       24    0.306917       0.306          0.000917
2017-01-02       24    0.306917       0.307         -0.000083
2017-01-03       24    0.306958       0.307         -0.000042
2017-01-04       24    0.306125       0.307         -0.000875
2017-01-05       24    0.305208       0.306         -0.000792
2017-01-06       24    0.303333       0.304         -0.000667
2017-01-07       24    0.300667       0.302         -0.001333
Jan2017 mean|mean-first|: mean=0.0290, max=55.2934
saved snotel_sample_daily.csv


In [8]:
train_path = REPO_ROOT / CFG["repo_paths"]["derived_train"]
tr = pd.read_csv(train_path, usecols=["station_id", "date", "soil_moisture_5cm"], low_memory=False)
tr["date"] = pd.to_datetime(tr["date"])
print(f"derived_8.4 train rows: {len(tr)}")
dup = tr.duplicated(subset=["station_id", "date"]).sum()
print(f"duplicate station-day rows: {dup}")
per = tr.groupby("station_id")["date"].agg(["min", "max", "count"])
print(per.to_string())

derived_8.4 train rows: 9803
duplicate station-day rows: 0
                             min        max  count
station_id                                        
BeaverPass_WA_990     2017-01-01 2020-12-31   1459
CayusePass_WA         2017-01-01 2020-12-31   1414
Darrington            2017-01-01 2020-12-31   1377
Paradise_WA           2017-01-01 2020-12-31   1459
Quinault              2017-01-01 2020-12-31   1447
SourdoughGulch_WA_985 2017-01-01 2020-12-31   1461
Spokane               2017-03-09 2020-12-24   1186


## Conclusion

We summarize the coverage verdict, the size of the averaging-definition effect, and what still needs a decision before ECE targets are compared to USCRN/SNOTEL targets.

In [9]:
print("hourly Value describe:")
print(hourly["Value"].describe().to_string())
bad = daily.sort_values("mean_minus_first", key=abs).tail(3)
print("largest |mean-first| days:")
print(bad.to_string(index=False))
print("USCRN daily01 reference: average of day/hourly values over 24h LST day (readme Note J).")
print("SNOTEL .stm finding: hourly rows; pipe averages sensors per timestamp, no daily mean in current code.")
print("ECE finding: Seattle-Time calendar-day mean; eval days have 16-24 distinct hours; Aug-01 fully missing.")

hourly Value describe:
count    58199.000000
mean         0.265192
std          5.500966
min          0.020000
25%          0.148000
50%          0.291000
75%          0.324000
max       1327.100000
largest |mean-first| days:
      date  n_hours  daily_mean  first_hour  mean_minus_first
2016-07-10       24    0.253583       0.151          0.102583
2020-09-26       24    0.175250       0.072          0.103250
2021-10-15       24   55.359417       0.066         55.293417
USCRN daily01 reference: average of day/hourly values over 24h LST day (readme Note J).
SNOTEL .stm finding: hourly rows; pipe averages sensors per timestamp, no daily mean in current code.
ECE finding: Seattle-Time calendar-day mean; eval days have 16-24 distinct hours; Aug-01 fully missing.


In [10]:
summary = {
    "eval_days_expected": 150,
    "eval_days_present": int(len(eval_cov)),
    "days_below_18h": int((eval_cov["n_hours"] < 18).sum()),
    "simple_vs_hourly_max_pp": float((ests_eval["simple_minus_hourly"].abs().max()) * 100),
    "simple_vs_snapshot_max_pp": float((ests_eval["simple_minus_snapshot"].abs().max()) * 100),
    "snotel_hourly_rows": int(len(hourly)),
    "train_duplicates": int(dup),
}
print(summary)
import json
with open(EXP_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("saved summary.json")

{'eval_days_expected': 150, 'eval_days_present': 150, 'days_below_18h': 3, 'simple_vs_hourly_max_pp': 0.36273426732366526, 'simple_vs_snapshot_max_pp': 3.8918705035971253, 'snotel_hourly_rows': 58199, 'train_duplicates': 0}
saved summary.json
